# Enjambre de Partículas
### Optimización: Programación Estructurada vs Enjambre de Partículas

**Función a minimizar**

$$f(x, y) = x^2 \left(4 - 2.1x^2 + \frac{x^4}{3}\right) + xy + y^2(-4 + 4y^2)$$

**Dominio:** $x \in [-2, 2]$, $y \in [-1, 1]$

Los mínimos globales conocidos son aproximadamente $f \approx -1.0316$ en $(0.0898, -0.7126)$ y $(-0.0898, 0.7126)$.

## 1. Importaciones y función objetivo

In [41]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import time
from random import random
from scipy.optimize import minimize

# Funcion
def funcion(x, y):

    sum1 = x**2 * (4 - 2.1*x**2 + x**4/3.0)
    sum2 = x * y
    sum3 = y**2 * (-4 + 4*y**2)
    return sum1 + sum2 + sum3

## 2. Método 1: Programación Estructurada

In [42]:
def optimizar_multistart(n_starts=30, seed=42):
    """
    Minimiza f(x,y) lanzando scipy.optimize.minimize desde
    n_starts puntos aleatorios y quedándose con el mejor resultado.
    """
    np.random.seed(seed)
    mejor = None

    for _ in range(n_starts):
        x0 = [np.random.uniform(-2, 2), np.random.uniform(-1, 1)]
        res = minimize(
            lambda v: funcion(v[0], v[1]),
            x0,
            method='L-BFGS-B',
            bounds=[(-2, 2), (-1, 1)]
        )
        if mejor is None or res.fun < mejor.fun:
            mejor = res

    return mejor.x[0], mejor.x[1], mejor.fun

t0 = time.time()
gx, gy, gval = optimizar_multistart()
t_struct = time.time() - t0

print(f"  x = {gx:.6f}")
print(f"  y = {gy:.6f}")
print(f"  f(x,y) = {gval:.6f}")
print(f"  Tiempo = {t_struct:.4f} s")

  x = 0.089842
  y = -0.712656
  f(x,y) = -1.031628
  Tiempo = 0.0663 s


## 3. Método 2 – Enjambre de Partículas (PSO)

In [43]:
from random import random as rnd

# Clase Particula
class Particula:
    # Atributos de clase (compartidos por todas las partículas)
    inercia = 1.4
    cognitiva = 2.0
    social = 2.0
    # Límites del espacio de soluciones
    infx = -10.0;  
    supx = 10.0
    infy = -10.0;  
    supy = 10.0
    # Factor de ajuste de la velocidad inicial
    ajusteV = 100.0

    def __init__(self):
        # Posición aleatoria dentro de los límites
        self.x = aleatorio(Particula.infx, Particula.supx)
        self.y = aleatorio(Particula.infy, Particula.supy)
        # Velocidad inicial aleatoria escalada
        self.vx = aleatorio(Particula.infx / Particula.ajusteV,
                            Particula.supx / Particula.ajusteV)
        self.vy = aleatorio(Particula.infy / Particula.ajusteV,
                            Particula.supy / Particula.ajusteV)
        # Mejor posición local
        self.xLoc     = self.x
        self.yLoc     = self.y
        self.valorLoc = funcion(self.x, self.y)

    def actualizaVelocidad(self, xGlob, yGlob):
        cogX = Particula.cognitiva * rnd() * (self.xLoc - self.x)
        socX = Particula.social    * rnd() * (xGlob    - self.x)
        self.vx = Particula.inercia * self.vx + cogX + socX

        cogY = Particula.cognitiva * rnd() * (self.yLoc - self.y)
        socY = Particula.social    * rnd() * (yGlob    - self.y)
        self.vy = Particula.inercia * self.vy + cogY + socY

    def actualizaPosicion(self):
        self.x = self.x + self.vx
        self.y = self.y + self.vy
        # Mantener dentro del espacio de soluciones
        self.x = max(self.x, Particula.infx)
        self.x = min(self.x, Particula.supx)
        self.y = max(self.y, Particula.infy)
        self.y = min(self.y, Particula.supy)
        # Si es mejor que el local, actualizar
        valor = funcion(self.x, self.y)
        if valor < self.valorLoc:
            self.xLoc     = self.x
            self.yLoc     = self.y
            self.valorLoc = valor


def aleatorio(inf, sup):
    return rnd() * (sup - inf) + inf


# Algoritmo PSO
def enjambreParticulas(particulas, iteraciones, reduccionInercia):
    """
    Mueve el enjambre durante las iteraciones indicadas.
    Devuelve coordenadas y valor del mínimo obtenido,
    más el historial del mejor valor global por iteración.
    """
    # Registra la mejor posición global inicial
    mejorParticula = min(particulas, key=lambda p: p.valorLoc)
    xGlob     = mejorParticula.xLoc
    yGlob     = mejorParticula.yLoc
    valorGlob = mejorParticula.valorLoc
    historial = [valorGlob]

    for _ in range(iteraciones):
        # Actualiza velocidad y posición de cada partícula
        for p in particulas:
            p.actualizaVelocidad(xGlob, yGlob)
            p.actualizaPosicion()

        # Actualiza el mínimo global
        mejorParticula = min(particulas, key=lambda p: p.valorLoc)
        if mejorParticula.valorLoc < valorGlob:
            xGlob     = mejorParticula.xLoc
            yGlob     = mejorParticula.yLoc
            valorGlob = mejorParticula.valorLoc

        historial.append(valorGlob)
        # Reduce la inercia de todas las partículas
        Particula.inercia *= reduccionInercia

    return xGlob, yGlob, valorGlob, historial


# Parámetros del problema 
nParticulas = 10
iteraciones = 100
redInercia = 0.9

# Resetear inercia antes de cada ejecución
Particula.inercia = 1.4

# Genera el conjunto inicial de partículas
particulas = [Particula() for _ in range(nParticulas)]

t0 = time.time()
px, py, pval, p_hist = enjambreParticulas(particulas, iteraciones, redInercia)
t_pso = time.time() - t0

print("---- Resultado de Enjambre de Partículas ----")
print(f"  x = {px:.6f}")
print(f"  y = {py:.6f}")
print(f"  f(x,y) = {pval:.6f}")
print(f"  Tiempo = {t_pso:.4f} s")

---- Resultado de Enjambre de Partículas ----
  x = 0.089842
  y = -0.712656
  f(x,y) = -1.031628
  Tiempo = 0.0045 s


## 4. Comparación de resultados

In [44]:
minimo_global = -1.031628

print("╔══════════════════════════════════════════════════════════════════╗")
print("║                COMPARATIVA DE MÉTODOS DE OPTIMIZACIÓN           ║")
print("╠══════════════════╦══════════╦══════════╦══════════╦═════════════╣")
print("║ Método           ║    x     ║    y     ║  f(x,y)  ║  Tiempo (s) ║")
print("╠══════════════════╬══════════╬══════════╬══════════╬═════════════╣")
print(f"║ Mínimo global    ║  0.0898  ║ -0.7126  ║ -1.03163 ║      —      ║")
print(f"║ Función          ║ {gx:8.4f} ║ {gy:8.4f} ║ {gval:8.5f} ║ {t_grid:11.4f} ║")
print(f"║ PSO (enjambre)   ║ {px:8.4f} ║ {py:8.4f} ║ {pval:8.5f} ║ {t_pso:11.4f} ║")
print("╚══════════════════╩══════════╩══════════╩══════════╩═════════════╝")

print("\nError absoluto respecto al mínimo global conocido:")
print(f"  Búsqueda rejilla : {abs(gval  - minimo_global):.2e}")
print(f"  PSO (enjambre)   : {abs(pval  - minimo_global):.2e}")

╔══════════════════════════════════════════════════════════════════╗
║                COMPARATIVA DE MÉTODOS DE OPTIMIZACIÓN           ║
╠══════════════════╦══════════╦══════════╦══════════╦═════════════╣
║ Método           ║    x     ║    y     ║  f(x,y)  ║  Tiempo (s) ║
╠══════════════════╬══════════╬══════════╬══════════╬═════════════╣
║ Mínimo global    ║  0.0898  ║ -0.7126  ║ -1.03163 ║      —      ║
║ Función          ║   0.0898 ║  -0.7127 ║ -1.03163 ║      0.1727 ║
║ PSO (enjambre)   ║   0.0898 ║  -0.7127 ║ -1.03163 ║      0.0045 ║
╚══════════════════╩══════════╩══════════╩══════════╩═════════════╝

Error absoluto respecto al mínimo global conocido:
  Búsqueda rejilla : 4.53e-07
  PSO (enjambre)   : 4.53e-07


## 5. Conclusión
En esta comparativa, tanto la optimización con funciones de SciPy como el método de enjambre de partículas (PSO) logran prácticamente el mismo resultado, alcanzando valores casi idénticos al mínimo global con un error absoluto muy bajo (4.53e-07). Sin embargo, la principal diferencia radica en el tiempo de ejecución: mientras que el método basado en funciones de SciPy tarda alrededor de 0.17 segundos, el PSO es significativamente más rápido (0.0045 s). Esto sugiere que, aunque ambos enfoques son igual de precisos en este caso, el PSO resulta más eficiente computacionalmente, especialmente útil en problemas donde el tiempo es crítico o la función objetivo es compleja.